## Youtube Analytics pipeline


objective: Data extraction and analysis from social media platform Youtube.


**Problem statement**

Videos are a fast growing medium where people communicate, share knowledge, showcase skills etc. YouTube is one of the biggest platforms which hosts videos. The YouTube platform hosts content from many different professions/arts/ cultures across the world.

People can express their opinion about the video in the form of likes, dislikes, comments which are features provided by the YouTube platform which provides the information on the sentiment about the video.

The objective involves the steps on programmatic data extraction from YouTube on which analysis can be conducted to understand various attributes related to a video.

**Steps to be performed**

1. Connect to the Youtube API using a Python client



> 1.a Create a YouTube API key





> 1.b Install the Google API python client



refer to the [supporting](https://developers.google.com/youtube/v3/getting-started) link on how to create YouTube API Key

Reference link : https://developers.google.com/youtube/v3/quickstart/python

In [ ]:
# <code block>
#1.a: We have created an API key using google cloud and executed the code
API_key = "PROTECTED"
#1.b: Code for installing the Google API python client
!pip install --upgrade google-api-python-client
!pip install --upgrade google-auth-oauthlib google-auth-httplib2

Search and extract the data



> Search videos related to the query string  “avatar movie”
(For this part, choose/search one video of your choice and perform data collection steps on that specific video )

> Output expected : ID, Snippet with following attributes Channel ID, Video Description, Channel Title, Video Title






Reference link:  https://developers.google.com/youtube/v3/docs/search/list

In [ ]:
# < code block >
#2.a: We first need to import some libraries to use the Youtube application
from apiclient.discovery import build
youtube = build('youtube', 'v3', developerKey=API_key)
#We later send the request for searching the specific query
request = youtube.search().list(q="avatar movie", part="snippet",type="video",maxResults = 1)

response = request.execute()
#we extracting the requested attributes:VideoID, Channel ID, Video Description, Channel Title, Video Title

ID = response['items'][0]['id']['videoId']
Channel_ID = response['items'][0]['snippet']['channelId']
Video_Description = response['items'][0]['snippet']['description']
Channel_Title = response['items'][0]['snippet']['channelTitle']
Video_Title = response['items'][0]['snippet']['title']

#we are displaying the snippet with the following attributes:VideoID Channel ID, Video Description, Channel Title, Video Title
print("ID :", ID),
print("Channel ID :",Channel_ID)
print("Video Description :",Video_Description)
print("Channel Title :",Channel_Title)
print("Video title :",Video_Title)

ID : X8SVkfbt8cs
Channel ID : UCHuSY3EIfBHdNLlsLepYmzg
Video Description : Avatar: The Way of Water reaches new heights and explores undiscovered depths as James Cameron returns to the world of ...
Channel Title : YouTube Movies
Video title : Avatar: The Way of Water



> We try to obtain the following statistics for query string “avatar movie” of top 50 videos sorted by relevance in the US region

> Output expected: video ID, title, no of views, no of likes,no of comments exported to CSV file






Reference link: https://developers.google.com/youtube/v3/docs/videos/list

In [ ]:
#<code block>
#2.b:
import pandas as pd

from apiclient.discovery import build
youtube = build('youtube', 'v3', developerKey=API_key)

#Fetching video data for most relevant 50 videos with the region set to the 'US'
request = youtube.search().list(q="avatar movie", part="snippet",type="video",maxResults = 50, regionCode="US")
response = request.execute()

# Create lists to store the data
video_ids = []
titles = []
views = []
likes = []
comments = []

# Iterate through each item in the response
for item in response['items']:
  video_id = item['id']['videoId']
  video_ids.append(video_id)

  title = item['snippet']['title']
  titles.append(title)

  # Get video statistics
  stats = youtube.videos().list(
    part="statistics",
    id=video_id
  ).execute()

  # Extracting the parameters: view, like and comment counts, handling potential missing keys/values
  view_count = stats['items'][0]['statistics'].get('viewCount', None) # Use get() to avoid KeyError
  views.append(view_count)

  like_count = stats['items'][0]['statistics'].get('likeCount', None)
  likes.append(like_count)

  comment_count = stats['items'][0]['statistics'].get('commentCount', None)
  comments.append(comment_count)

# Creating a Pandas DataFrame
df = pd.DataFrame({
  'video_id': video_ids,
  'title': titles,
  'views': views,
  'likes': likes,
  'comments': comments
})

# Export to CSV as requested
df.to_csv('youtube_data.csv', index=False)

# Display the DataFrame to view data
df

,video_id,title,views,likes,comments
0,nxjOFZqtLJc,AVATAR 3 Official Trailer (2024) | 20th Centur...,2693263,26973,640
1,3G6J-ITWekA,Jake Wakes up in his Avatar Body - AVATAR (4k ...,6533427,27561,767
2,d9MyW72ELq0,Avatar: The Way of Water | Official Trailer,59236291,1040751,42699
3,gGw36otjMUw,AVATAR 2 In English ( Full Movie),316313,1172,16
4,fbgbKYeVfmU,AVATAR 2 : Making of Kissing Scene of Avatar M...,234271,1367,72
5,X8SVkfbt8cs,Avatar: The Way of Water,None,22626,1833
6,ofvrOdrO9gY,Neytiri saves Jake (Ending Scene) - AVATAR (4k...,2066854,24429,1113
7,L2d3SocB7as,Avatar: The Way of Water (2022) - Neteyam death,456985,None,529
8,WDFr1O_GeAQ,Avatar (2009) - Best Scenes,2747547,20201,638
9,6pHZmBin4Xc,Toruk Macto - AVATAR (4k Movie Clip),4763270,33534,741





## 📈 Analysis

### 🔥 Top Videos by Comment Count

Videos with higher comment counts indicate stronger audience engagement.

*   List item
*   List item



In [ ]:
# < code block >
import pandas as pd
df = pd.read_csv('youtube_data.csv')
# Sort by comments in descending order
sorted_df = df.sort_values(by='comments', ascending=False)
# Get the top 10 videos
top_10_videos = sorted_df.head(10)
#printing only the top 10 videos that have highest comments
top_10_videos

,video_id,title,views,likes,comments
2,d9MyW72ELq0,Avatar: The Way of Water | Official Trailer,59236291.0,1040751.0,42699
22,a8Gx8wiNbs8,Avatar: The Way of Water | Official Teaser Tra...,28306747.0,681687.0,28933
29,bDHD1ueL4a4,The Weeknd - Nothing Is Lost (You Give Me Stre...,30959748.0,657160.0,9733
25,RGx8rYbRVR4,Why People Hate Avatar: A Lesson In Lazy Comme...,625110.0,33914.0,5206
38,doDrnJkgf-s,Zoe Saldana Emotional Avatar Scene Behind The ...,15958317.0,1236225.0,5099
47,f5Zx8iPek5I,Avatar 3 Will Introduce The Dark Side🔥 Of Na&#...,8131313.0,366276.0,4050
48,0sJeBiUCIt4,AVATAR Clip - Final Battle (2009) James Cameron,33958195.0,197924.0,3823
46,mRrKdgpZ6kE,AGAINST JAKE..!??😲😲 | AVATAR 3 #shorts #avatar,3683351.0,NaN,2300
5,X8SVkfbt8cs,Avatar: The Way of Water,NaN,22626.0,1833
31,oFErWcXJLdw,TRAINING TO BE IN THE NEXT AVATAR MOVIE,3135449.0,53434.0,1763





## 💬 Comment Extraction

We retrieve comments from the top-performing videos to understand audience sentiment and interaction.




In [ ]:
# < code block >
import pandas as pd
from apiclient.discovery import build
from pprint import pprint # Import pprint for pretty printing

for video_id in top_10_videos['video_id']:
    request = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        textFormat="plainText" # Retrieve comments as plain text
    )
    response1 = request.execute()

    print("Comments for video:", video_id)
    pprint(response1) # Print the response in a more readable format


Streaming output truncated to the last 5000 lines.
            'kind': 'youtube#commentThread',
            'snippet': {'canReply': True,
                        'channelId': 'UC4qGmRZ7aLOLfVsSdj5Se2A',
                        'isPublic': True,
                        'topLevelComment': {'etag': 'tNOd22XLLAsuIfwUSh6TYkBGzRg',
                                            'id': 'Ugw5wDehFkQFTuooLcB4AaABAg',
                                            'kind': 'youtube#comment',
                                            'snippet': {'authorChannelId': {'value': 'UCYDJHKYk2X_QiqmMzz4B0sg'},
                                                        'authorChannelUrl': 'http://www.youtube.com/@lovelymama_',
                                                        'authorDisplayName': '@lovelymama_',
                                                        'authorProfileImageUrl': 'https://yt3.ggpht.com/ytc/AIdro_mConJRZ_6lALcQINW-T8S9t6YdMT9NnehgQW6uMH8bTeyK=s48-c-k-c0x00ffffff-no-rj',
          



> Now we write a program to export the output of previous step in JSON file format and submit the file as part of the assignment *italicized text*



In [ ]:
# < code block >
import json

#  from the comment data from 3.b
with open('response1.json', 'w') as f:
    json.dump(response1, f, indent=4)

>3.d Write a function to get  the likes vs views ratio of the top 10 videos obtained in 3.a with the highest comments

## ⚡ Engagement Metrics

We compute the likes-to-views ratio to evaluate audience engagement quality.




In [ ]:
def likes_views_ratio(df):
  """
  Objective:
    Calculates the likes to views ratio for each video in the DataFrame.

  Arguments of function:
    df: A Pandas DataFrame containing video data with 'likes' and 'views' columns.

  Return value of function:
    A Pandas Series containing the likes to views ratios.
  """
  # Handle potential division by zero using .loc to avoid dataframe error of manipulating existing dataframe
  df.loc[:, 'ratio'] = df['likes'] / df['views'].replace(0, 1)
  return df['ratio']

# Assuming 'top_10_videos' is the DataFrame from 3.a
ratios = likes_views_ratio(top_10_videos)
ratios

2     0.017569
22    0.024082
29    0.021226
25    0.054253
38    0.077466
47    0.045045
48    0.005828
46         NaN
5          NaN
31    0.017042
Name: ratio, dtype: float64

## 🧠 Key Insights

- Videos with high comments tend to have stronger audience engagement
- Likes-to-views ratio provides a better measure of engagement than views alone
- Comments offer valuable qualitative insights into audience sentiment

---

## 🚀 Future Work

- Perform sentiment analysis on comments
- Build a dashboard for real-time insights
- Expand analysis to multiple categories